# Three Ways to Mark Change

## Comparing fixed thresholds with adaptive thresholds

Let \(x_t=100\log P_t\), so that \(\Delta x_t=r_t\) is the daily log return in percentage points. A symmetric cumulative sum keeps

$$
g_t^+=\max\{0,g_{t-1}^++\Delta x_t\},
\qquad
g_t^-=\min\{0,g_{t-1}^-+\Delta x_t\}.
$$

Here, \(g_t^+\) accumulates positive moves and \(g_t^-\) accumulates negative moves. An event is recorded when \(g_t^+>h_t\) or \(g_t^-<-h_t\), after which the crossed sum is reset. The fixed rule uses one threshold \(h\). The changing rule uses

$$
h_t=3\widehat\sigma_{t-1}^{(63)},
$$

where \(\widehat\sigma_{t-1}^{(63)}\) is the population standard deviation of the previous 63 daily returns. The threshold is therefore known before the return on date \(t\). Neither rule is presented as optimal.

In [1]:
from pathlib import Path
import hashlib
import sys
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.stats import norm
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset

import RiskLabAI
from RiskLabAI.backtest.adaptation import prequential_adaptation
from RiskLabAI.data.labeling import cusum_filter_events_dynamic_threshold, symmetric_cusum_filter
from RiskLabAI.features.structural_breaks.pelt_change_points import pelt_change_points
from river.drift import ADWIN

assert RiskLabAI.__version__ == "3.2.0"

warnings.filterwarnings("ignore", message="New behaviour in v1.1.5")

DATA_PATH = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
FIGURES = BOOK_ROOT / "outputs" / "figures" / "13_regime_adaptation"
TABLES = BOOK_ROOT / "outputs" / "tables" / "13_regime_adaptation"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "13_regime_adaptation"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return "0"
        if number.is_integer():
            return f"{int(number):,}"
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f"{number:.6g}"
        return f"{number:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)


def load_public_data():
    expected_hash = "2E988948FFCB108DA4902DFEDAA5DF756E7E102DC523A99A8916A0533291F2D7"
    assert DATA_PATH.is_file()
    assert sha256(DATA_PATH) == expected_hash
    raw = pd.read_csv(DATA_PATH)
    raw["Date"] = pd.to_datetime(raw["observation_date"])
    raw["NASDAQ Composite"] = pd.to_numeric(raw["NASDAQCOM"], errors="coerce")
    data = raw.dropna(subset=["NASDAQ Composite"]).sort_values("Date").reset_index(drop=True)
    data["Daily log return (%)"] = 100 * np.log(data["NASDAQ Composite"] / data["NASDAQ Composite"].shift(1))
    data["Log index level"] = 100 * np.log(data["NASDAQ Composite"])
    return data


def prepare_sequential_data(data, min_samples=756):
    frame = data[["Date", "Daily log return (%)"]].copy()
    lagged = frame["Daily log return (%)"].shift(1)
    frame["Previous return (%)"] = lagged
    frame["Previous absolute return (%)"] = lagged.abs()
    frame["Trailing 5 day mean (%)"] = lagged.rolling(5).mean()
    frame["Trailing 21 day volatility (%)"] = lagged.rolling(21).std(ddof=0)
    frame["Trailing 63 day volatility (%)"] = lagged.rolling(63).std(ddof=0)
    frame = frame.dropna().reset_index(drop=True)
    threshold = float(frame.loc[: min_samples - 1, "Daily log return (%)"].abs().quantile(0.80))
    frame["Large move"] = (frame["Daily log return (%)"].abs() > threshold).astype(int)
    return frame, threshold


def feature_columns():
    return [
        "Previous return (%)",
        "Previous absolute return (%)",
        "Trailing 5 day mean (%)",
        "Trailing 21 day volatility (%)",
        "Trailing 63 day volatility (%)",
    ]


def make_estimator():
    return Pipeline([
        ("standardization", StandardScaler()),
        ("logistic regression", LogisticRegression(C=1, class_weight="balanced", random_state=314159, max_iter=1000)),
    ])


def zero_one_loss(actual, prediction):
    return float(actual != prediction)


def run_adaptive(features, targets, delta=0.01, min_samples=756, window_size=756):
    return prequential_adaptation(
        features,
        targets,
        make_estimator(),
        loss_function=zero_one_loss,
        window_size=window_size,
        min_samples=min_samples,
        delta=delta,
    )


def run_static(features, targets, min_samples=756):
    predictions = np.full(len(targets), np.nan)
    losses = np.full(len(targets), np.nan)
    model = clone(make_estimator()).fit(features[:min_samples], targets[:min_samples])
    predictions[min_samples:] = model.predict(features[min_samples:])
    losses[min_samples:] = (predictions[min_samples:] != targets[min_samples:]).astype(float)
    return {"predictions": predictions, "losses": losses, "refit_indices": np.array([min_samples - 1], dtype=int)}


def run_periodic(features, targets, min_samples=756, window_size=756, interval=252):
    predictions = np.full(len(targets), np.nan)
    losses = np.full(len(targets), np.nan)
    refits = [min_samples - 1]
    model = clone(make_estimator()).fit(features[:min_samples], targets[:min_samples])
    for index in range(min_samples, len(targets)):
        predictions[index] = model.predict(features[index : index + 1])[0]
        losses[index] = float(predictions[index] != targets[index])
        if (index + 1 - min_samples) % interval == 0:
            begin = max(0, index + 1 - window_size)
            model = clone(make_estimator()).fit(features[begin : index + 1], targets[begin : index + 1])
            refits.append(index)
    return {"predictions": predictions, "losses": losses, "refit_indices": np.array(refits, dtype=int)}

In [2]:
data = load_public_data()
model_frame, large_move_threshold = prepare_sequential_data(data)
min_samples = 756
initial_returns = model_frame.loc[: min_samples - 1, "Daily log return (%)"]
fixed_threshold = float(3 * initial_returns.std(ddof=0))

log_index = pd.Series(data["Log index level"].to_numpy(), index=data["Date"])
indexed_returns = pd.Series(data["Daily log return (%)"].to_numpy(), index=data["Date"])
changing_threshold = 3 * indexed_returns.rolling(63).std(ddof=0).shift(1)
fixed_events = symmetric_cusum_filter(log_index, fixed_threshold)
changing_events = cusum_filter_events_dynamic_threshold(log_index, changing_threshold)

settings = pd.DataFrame([
    {"Setting": "Input series", "Fixed threshold rule": "100 times log index level", "Changing threshold rule": "100 times log index level"},
    {"Setting": "Threshold multiplier", "Fixed threshold rule": 3, "Changing threshold rule": 3},
    {"Setting": "Volatility sample", "Fixed threshold rule": "First 756 complete model rows", "Changing threshold rule": "Previous 63 returns"},
    {"Setting": "Threshold timing", "Fixed threshold rule": "Frozen before evaluation", "Changing threshold rule": "Known through the prior date"},
    {"Setting": "Reset", "Fixed threshold rule": "Crossed cumulative sum", "Changing threshold rule": "Crossed cumulative sum"},
])
save_table(settings, "cusum_settings", "Settings for the fixed and changing CUSUM comparisons.", "tab:chapter14-cusum-settings", fit=True)

summary = pd.DataFrame([
    {"Rule": "Fixed threshold", "Events": len(fixed_events), "First event": fixed_events.min(), "Last event": fixed_events.max(), "Threshold description": fixed_threshold},
    {"Rule": "Changing threshold", "Events": len(changing_events), "First event": changing_events.min(), "Last event": changing_events.max(), "Threshold description": "Three times prior 63 day volatility"},
])
save_table(summary, "cusum_summary", "CUSUM event counts under the two predeclared threshold rules.", "tab:chapter14-cusum-summary", fit=True)

years = range(int(data["Date"].dt.year.min()), int(data["Date"].dt.year.max()) + 1)
year_counts = pd.DataFrame({
    "Year": list(years),
    "Fixed threshold events": [int((fixed_events.year == year).sum()) for year in years],
    "Changing threshold events": [int((changing_events.year == year).sum()) for year in years],
})
save_table(year_counts, "cusum_event_counts_by_year", "CUSUM event counts by calendar year.", "tab:chapter14-cusum-event-counts")

event_excerpt_rows = []
for rule, event_index in [("Fixed threshold", fixed_events), ("Changing threshold", changing_events)]:
    selected = list(event_index[:5]) + list(event_index[-5:])
    for position, date in enumerate(selected, start=1):
        event_excerpt_rows.append({"Rule": rule, "Excerpt position": position, "Event date": date})
event_excerpt = pd.DataFrame(event_excerpt_rows)
save_table(event_excerpt, "cusum_event_dates_excerpt", "First and last five dates from each CUSUM event list.", "tab:chapter14-cusum-event-dates")

event_data = data[["Date", "Log index level"]].copy()
event_data["Fixed threshold event"] = event_data["Date"].isin(fixed_events)
event_data["Changing threshold event"] = event_data["Date"].isin(changing_events)
event_data.to_csv(FIGURE_DATA / "cusum_events.csv", index=False)
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.2), sharex=True, constrained_layout=True)
for axis, column, color, title in [
    (axes[0], "Fixed threshold event", PALETTE["amber"], "Fixed Threshold Events"),
    (axes[1], "Changing threshold event", PALETTE["green"], "Changing Threshold Events"),
]:
    axis.plot(event_data["Date"], event_data["Log index level"], color=PALETTE["cyan"], linewidth=1.05)
    marked = event_data[column]
    axis.scatter(event_data.loc[marked, "Date"], event_data.loc[marked, "Log index level"], color=color, s=8, alpha=0.75, label=title)
    axis.set_ylabel("100 log index")
    axis.legend(loc="upper left")
    finish_axis(axis)
axes[0].set_title("CUSUM Events on the Public Log Price")
axes[1].set_xlabel("Date")
save_figure(figure, "cusum_events_log_price")
plt.close(figure)

threshold_data = pd.DataFrame({"Date": changing_threshold.index, "Changing threshold": changing_threshold.to_numpy()}).dropna()
threshold_data["Fixed threshold"] = fixed_threshold
threshold_data.to_csv(FIGURE_DATA / "cusum_dynamic_threshold.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.3), constrained_layout=True)
axis.plot(threshold_data["Date"], threshold_data["Changing threshold"], color=PALETTE["green"], linewidth=1.15, label="Changing threshold")
axis.axhline(fixed_threshold, color=PALETTE["amber"], linestyle="--", linewidth=1.5, label="Fixed threshold")
axis.set_xlabel("Date")
axis.set_ylabel("Threshold in return percentage points")
axis.set_title("Fixed and Prior 63 Day CUSUM Thresholds")
axis.legend()
finish_axis(axis)
save_figure(figure, "cusum_dynamic_threshold")
plt.close(figure)

year_counts.to_csv(FIGURE_DATA / "cusum_annual_event_counts.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.4), constrained_layout=True)
axis.plot(year_counts["Year"], year_counts["Fixed threshold events"], color=PALETTE["amber"], marker="o", markersize=3.8, linewidth=1.4, label="Fixed threshold")
axis.plot(year_counts["Year"], year_counts["Changing threshold events"], color=PALETTE["green"], marker="s", markersize=3.5, linewidth=1.4, label="Changing threshold")
axis.set_xlabel("Year")
axis.set_ylabel("Event count")
axis.set_title("CUSUM Event Counts Depend on the Threshold Rule")
axis.legend()
finish_axis(axis)
save_figure(figure, "cusum_annual_event_counts")
plt.close(figure)

,Setting,Fixed threshold rule,Changing threshold rule
0,Input series,100 times log index level,100 times log index level
1,Threshold multiplier,3,3
2,Volatility sample,First 756 complete model rows,Previous 63 returns
3,Threshold timing,Frozen before evaluation,Known through the prior date
4,Reset,Crossed cumulative sum,Crossed cumulative sum


,Rule,Events,First event,Last event,Threshold description
0,Fixed threshold,"1,599",1994-01-14,2025-12-19,2.619751
1,Changing threshold,986,1994-04-05,2025-12-23,Three times prior 63 day volatility


,Year,Fixed threshold events,Changing threshold events
0,"1,994",23,23
1,"1,995",33,41
2,"1,996",34,30
3,"1,997",45,36
4,"1,998",62,37
5,"1,999",81,34
6,"2,000",121,41
7,"2,001",98,28
8,"2,002",97,31
9,"2,003",63,29


,Rule,Excerpt position,Event date
0,Fixed threshold,1,1994-01-14
1,Fixed threshold,2,1994-02-04
2,Fixed threshold,3,1994-03-16
3,Fixed threshold,4,1994-03-28
4,Fixed threshold,5,1994-03-30
5,Fixed threshold,6,2025-11-20
6,Fixed threshold,7,2025-11-24
7,Fixed threshold,8,2025-12-04
8,Fixed threshold,9,2025-12-17
9,Fixed threshold,10,2025-12-19


## ADWIN on Bounded Prediction Mistakes

After the 63 day feature warmup, the first 756 complete rows define the initial sample. Let \(c\) be the 80th percentile of the absolute returns in that sample. The target is

$$
y_t=\mathbf{1}\{|r_t|>c\}.
$$

Here, \(y_t=1\) marks a large absolute return under the frozen threshold \(c\), while \(y_t=0\) marks the other dates. The model available before observing \(y_t\) produces \(\widehat y_t\), and its bounded mistake is

$$
L_t=\mathbf{1}\{\widehat y_t\ne y_t\}.
$$

The loss \(L_t\) is zero for a correct class and one for an error. ADWIN receives this value only after the prediction has been made.

The original ADWIN construction considers a split \(W=W_0W_1\) of the current window. With \(n_0\) and \(n_1\) observations in the two parts, define

$$
m=\left(\frac{1}{n_0}+\frac{1}{n_1}\right)^{-1},
\qquad
\delta'=\frac{\delta}{n},
\qquad
\epsilon_{\mathrm{cut}}
=\sqrt{\frac{1}{2m}\log\frac{4}{\delta'}}.
$$

Here, \(n=n_0+n_1\), \(m\) is the harmonic combination of the subwindow sizes, \(\delta\) is the confidence parameter, and \(\epsilon_{\mathrm{cut}}\) is the stated cut threshold. A difference in subwindow means larger than the threshold is evidence of change. The notebook uses River's efficient ADWIN implementation with the fixed parameter \(\delta=0.01\).

In [3]:
features = model_frame[feature_columns()].to_numpy()
targets = model_frame["Large move"].to_numpy()
adaptive = run_adaptive(features, targets, delta=0.01)
evaluation_dates = model_frame.loc[min_samples:, "Date"].reset_index(drop=True)
evaluation_losses = adaptive["losses"][min_samples:]

feature_table = pd.DataFrame([
    {"Feature": "Previous return", "Definition": "Return on date t minus 1", "Latest information date": "t minus 1"},
    {"Feature": "Previous absolute return", "Definition": "Absolute return on date t minus 1", "Latest information date": "t minus 1"},
    {"Feature": "Trailing 5 day mean", "Definition": "Mean of the five returns ending at t minus 1", "Latest information date": "t minus 1"},
    {"Feature": "Trailing 21 day volatility", "Definition": "Population standard deviation of 21 returns ending at t minus 1", "Latest information date": "t minus 1"},
    {"Feature": "Trailing 63 day volatility", "Definition": "Population standard deviation of 63 returns ending at t minus 1", "Latest information date": "t minus 1"},
])
save_table(feature_table, "sequential_feature_definitions", "Predictors and their information dates.", "tab:chapter14-sequential-features", fit=True)

target_summary = pd.DataFrame([
    {"Quantity": "Complete rows after feature warmup", "Value": len(model_frame)},
    {"Quantity": "Initial rows", "Value": min_samples},
    {"Quantity": "First initial date", "Value": model_frame.loc[0, "Date"]},
    {"Quantity": "Last initial date", "Value": model_frame.loc[min_samples - 1, "Date"]},
    {"Quantity": "Large move threshold (%)", "Value": large_move_threshold},
    {"Quantity": "Initial large move share", "Value": targets[:min_samples].mean()},
    {"Quantity": "Evaluation rows", "Value": len(targets) - min_samples},
    {"Quantity": "Evaluation large move share", "Value": targets[min_samples:].mean()},
])
save_table(target_summary, "target_summary", "Sequential target and frozen initial sample.", "tab:chapter14-target-summary")

adwin_settings = pd.DataFrame([
    {"Setting": "Confidence parameter", "Value": 0.01},
    {"Setting": "Initial observations", "Value": min_samples},
    {"Setting": "Maximum refit window", "Value": 756},
    {"Setting": "Loss range", "Value": "Zero to one"},
    {"Setting": "Estimator", "Value": "Standardized logistic regression"},
    {"Setting": "Inverse regularization strength", "Value": 1},
    {"Setting": "Class weights", "Value": "Balanced"},
    {"Setting": "Fixed seed", "Value": 314159},
])
save_table(adwin_settings, "adwin_settings", "Fixed settings for the sequential ADWIN calculation.", "tab:chapter14-adwin-settings")

alarm_dates = model_frame.loc[adaptive["alarm_indices"], ["Date"]].copy()
alarm_dates.insert(0, "Alarm number", np.arange(1, len(alarm_dates) + 1))
alarm_dates["Loss on alarm date"] = adaptive["losses"][adaptive["alarm_indices"]]
save_table(alarm_dates, "adwin_alarm_dates", "ADWIN alarm dates in the fixed public calculation.", "tab:chapter14-adwin-alarm-dates")

prefix_errors = []
for endpoint in [2500, 5000]:
    prefix = run_adaptive(features[:endpoint], targets[:endpoint], delta=0.01)
    prefix_errors.append(np.nanmax(np.abs(adaptive["predictions"][:endpoint] - prefix["predictions"])))
altered_targets = targets[:2501].copy()
altered_targets[2000] = 1 - altered_targets[2000]
unaltered = run_adaptive(features[:2501], targets[:2501], delta=0.01)
altered = run_adaptive(features[:2501], altered_targets, delta=0.01)

detector = ADWIN(delta=0.01)
widths = []
replayed_alarm_indices = []
for index in range(min_samples, len(targets)):
    detector.update(float(adaptive["losses"][index]))
    widths.append(float(detector.width))
    if detector.drift_detected:
        replayed_alarm_indices.append(index)

temporal_checks = pd.DataFrame([
    {"Check": "Predictions match when later rows are removed", "Maximum absolute error": max(prefix_errors), "Tolerance": 0, "Result": max(prefix_errors) == 0},
    {"Check": "Changing the current outcome leaves its own prediction unchanged", "Maximum absolute error": abs(unaltered["predictions"][2000] - altered["predictions"][2000]), "Tolerance": 0, "Result": unaltered["predictions"][2000] == altered["predictions"][2000]},
    {"Check": "Every stored loss is zero or one", "Maximum absolute error": 0, "Tolerance": 0, "Result": bool(np.isin(evaluation_losses, [0, 1]).all())},
    {"Check": "ADWIN replay gives the stored alarm dates", "Maximum absolute error": 0, "Tolerance": 0, "Result": bool(np.array_equal(np.array(replayed_alarm_indices, dtype=int), adaptive["alarm_indices"]))},
    {"Check": "Alarm refits start with the next prediction", "Maximum absolute error": 0, "Tolerance": 0, "Result": bool(np.array_equal(adaptive["refit_indices"][1:], adaptive["alarm_indices"]))},
])
save_table(temporal_checks, "temporal_order_checks", "Checks that preserve prediction before observation and later-only refitting.", "tab:chapter14-temporal-order-checks", fit=True)

loss_data = pd.DataFrame({
    "Date": evaluation_dates,
    "Mistake": evaluation_losses,
})
loss_data["Rolling 126 day mean loss"] = loss_data["Mistake"].rolling(126, min_periods=30).mean()
loss_data["ADWIN alarm"] = loss_data["Date"].isin(alarm_dates["Date"])
loss_data.to_csv(FIGURE_DATA / "adwin_loss_alarms.csv", index=False)
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.0), sharex=True, constrained_layout=True)
mistakes = loss_data["Mistake"] == 1
axes[0].scatter(loss_data.loc[mistakes, "Date"], loss_data.loc[mistakes, "Mistake"], color=PALETTE["amber"], s=5, alpha=0.55, label="Prediction mistake")
for date in alarm_dates["Date"]:
    axes[0].axvline(date, color=PALETTE["rose"], linewidth=0.9, alpha=0.7)
axes[0].set_ylim(-0.05, 1.12)
axes[0].set_ylabel("Bounded loss")
axes[0].set_title("Sequential Prediction Mistakes and ADWIN Alarms")
axes[0].legend()
axes[1].plot(loss_data["Date"], loss_data["Rolling 126 day mean loss"], color=PALETTE["cyan"], linewidth=1.35)
for date in alarm_dates["Date"]:
    axes[1].axvline(date, color=PALETTE["rose"], linewidth=0.9, alpha=0.7)
axes[1].set_ylabel("Rolling mean loss")
axes[1].set_xlabel("Date")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "bounded_loss_adwin_alarms")
plt.close(figure)

width_data = pd.DataFrame({
    "Date": evaluation_dates,
    "ADWIN window width": widths,
    "ADWIN alarm": evaluation_dates.isin(alarm_dates["Date"]),
})
width_data.to_csv(FIGURE_DATA / "adwin_window_width.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.2), constrained_layout=True)
axis.plot(width_data["Date"], width_data["ADWIN window width"], color=PALETTE["violet"], linewidth=1.25)
for date in alarm_dates["Date"]:
    axis.axvline(date, color=PALETTE["rose"], linewidth=0.9, alpha=0.7)
axis.set_xlabel("Date")
axis.set_ylabel("Detector window width")
axis.set_title("ADWIN Window Width Contracts When an Alarm Is Raised")
finish_axis(axis)
save_figure(figure, "adwin_window_width")
plt.close(figure)

,Feature,Definition,Latest information date
0,Previous return,Return on date t minus 1,t minus 1
1,Previous absolute return,Absolute return on date t minus 1,t minus 1
2,Trailing 5 day mean,Mean of the five returns ending at t minus 1,t minus 1
3,Trailing 21 day volatility,Population standard deviation of 21 returns en...,t minus 1
4,Trailing 63 day volatility,Population standard deviation of 63 returns en...,t minus 1


,Quantity,Value
0,Complete rows after feature warmup,"7,991"
1,Initial rows,756
2,First initial date,1994-04-05
3,Last initial date,1997-04-01
4,Large move threshold (%),1.033089
5,Initial large move share,0.199735
6,Evaluation rows,"7,235"
7,Evaluation large move share,0.378991


,Setting,Value
0,Confidence parameter,0.01
1,Initial observations,756
2,Maximum refit window,756
3,Loss range,Zero to one
4,Estimator,Standardized logistic regression
5,Inverse regularization strength,1
6,Class weights,Balanced
7,Fixed seed,"314,159"


,Alarm number,Date,Loss on alarm date
1587,1,2000-07-18,0
2323,2,2003-06-25,1
2771,3,2005-04-06,0
3443,4,2007-12-05,1
3731,5,2009-01-28,0
4979,6,2014-01-13,1
5555,7,2016-04-27,1
6259,8,2019-02-13,1
7283,9,2023-03-08,1


,Check,Maximum absolute error,Tolerance,Result
0,Predictions match when later rows are removed,0,0,Pass
1,Changing the current outcome leaves its own pr...,0,0,Pass
2,Every stored loss is zero or one,0,0,Pass
3,ADWIN replay gives the stored alarm dates,0,0,Pass
4,Alarm refits start with the next prediction,0,0,Pass


## Retrospective PELT Segments

Let the monthly loss observations be \(z_1,\ldots,z_T\). For ordered change points \(0=\tau_0<\tau_1<\cdots<\tau_m<\tau_{m+1}=T\), PELT solves

$$
\min_{m,\tau_1,\ldots,\tau_m}
\left\{
\sum_{j=0}^{m}
C\!\left(z_{\tau_j+1:\tau_{j+1}}\right)
+m\beta
\right\}.
$$

Here, \(C\) is the Gaussian segment cost, \(m\) is the number of interior change points, and \(\beta=3\log T\) is the penalty used in this notebook. Each segment must contain at least twelve months, and every monthly candidate date is considered. PELT sees the complete monthly loss series, so its dates are retrospective descriptions and never trigger a prediction or refit.

In [4]:
loss_series = pd.Series(evaluation_losses, index=evaluation_dates)
monthly_loss = loss_series.groupby(loss_series.index.to_period("M")).mean()
change_indices = pelt_change_points(monthly_loss.to_numpy(), model="normal", penalty_multiplier=3, min_size=12, jump=1)
boundaries = [0, *change_indices, len(monthly_loss)]

pelt_settings = pd.DataFrame([
    {"Setting": "Input", "Value": "Monthly mean ADWIN policy loss"},
    {"Setting": "Months", "Value": len(monthly_loss)},
    {"Setting": "Segment cost", "Value": "Gaussian mean and variance cost"},
    {"Setting": "Penalty multiplier", "Value": 3},
    {"Setting": "Penalty", "Value": 3 * np.log(len(monthly_loss))},
    {"Setting": "Minimum segment length", "Value": 12},
    {"Setting": "Candidate date step", "Value": 1},
    {"Setting": "Decision use", "Value": "Retrospective only"},
])
save_table(pelt_settings, "pelt_settings", "Fixed settings for the retrospective PELT comparison.", "tab:chapter14-pelt-settings")

change_dates = pd.DataFrame([
    {"Change number": number, "First month after change": monthly_loss.index[index].to_timestamp(), "Monthly position": index}
    for number, index in enumerate(change_indices, start=1)
])
save_table(change_dates, "pelt_change_dates", "Retrospective PELT change dates in monthly prediction loss.", "tab:chapter14-pelt-change-dates")

segment_rows = []
segment_labels = np.empty(len(monthly_loss), dtype=int)
for segment, (begin, end) in enumerate(zip(boundaries[:-1], boundaries[1:]), start=1):
    values = monthly_loss.iloc[begin:end]
    segment_labels[begin:end] = segment
    segment_rows.append({
        "Segment": segment,
        "First month": values.index.min().to_timestamp(),
        "Last month": values.index.max().to_timestamp(),
        "Months": len(values),
        "Mean loss": values.mean(),
        "Loss standard deviation": values.std(ddof=0),
    })
segment_table = pd.DataFrame(segment_rows)
save_table(segment_table, "pelt_segment_summary", "Retrospective monthly loss segments returned by PELT.", "tab:chapter14-pelt-segment-summary")

monthly_data = pd.DataFrame({
    "Month": monthly_loss.index.to_timestamp(),
    "Monthly mean loss": monthly_loss.to_numpy(),
    "Segment": segment_labels,
})
monthly_data["Segment mean loss"] = monthly_data.groupby("Segment")["Monthly mean loss"].transform("mean")
monthly_data.to_csv(FIGURE_DATA / "pelt_monthly_segments.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.4), constrained_layout=True)
axis.plot(monthly_data["Month"], monthly_data["Monthly mean loss"], color=PALETTE["cyan"], linewidth=1.0, alpha=0.85, label="Monthly mean loss")
axis.step(monthly_data["Month"], monthly_data["Segment mean loss"], where="post", color=PALETTE["amber"], linewidth=2.2, label="PELT segment mean")
for date in change_dates["First month after change"]:
    axis.axvline(date, color=PALETTE["rose"], linestyle="--", linewidth=1.1)
axis.set_xlabel("Month")
axis.set_ylabel("Mean prediction loss")
axis.set_title("Retrospective PELT Segments of Monthly Prediction Loss")
axis.legend()
finish_axis(axis)
save_figure(figure, "monthly_pelt_segments")
plt.close(figure)

,Setting,Value
0,Input,Monthly mean ADWIN policy loss
1,Months,345
2,Segment cost,Gaussian mean and variance cost
3,Penalty multiplier,3
4,Penalty,17.530633
5,Minimum segment length,12
6,Candidate date step,1
7,Decision use,Retrospective only


,Change number,First month after change,Monthly position
0,1,2004-11-01,91
1,2,2007-07-01,123
2,3,2008-09-01,137


,Segment,First month,Last month,Months,Mean loss,Loss standard deviation
0,1,1997-04-01,2004-10-01,91,0.404263,0.161671
1,2,2004-11-01,2007-06-01,32,0.18701,0.095341
2,3,2007-07-01,2008-08-01,14,0.510602,0.133343
3,4,2008-09-01,2025-12-01,208,0.308935,0.155619


## Why the Dates Differ

The three calculations mark different objects. CUSUM accumulates moves in the public log price. ADWIN monitors the bounded sequence of prediction mistakes and issues an online alarm after each observed loss. PELT partitions the completed monthly loss history. Agreement is possible, but equality of dates is not expected and is not a requirement for validity.

In [5]:
detector_summary = pd.DataFrame([
    {"Method": "Fixed threshold CUSUM", "Input": "Daily log price moves", "Timing": "Online event", "Default event count": len(fixed_events), "Role": "Marks cumulative price moves"},
    {"Method": "Changing threshold CUSUM", "Input": "Daily log price moves", "Timing": "Online event", "Default event count": len(changing_events), "Role": "Uses prior 63 day volatility"},
    {"Method": "ADWIN", "Input": "Daily prediction mistakes", "Timing": "Online alarm", "Default event count": len(alarm_dates), "Role": "May trigger a later refit"},
    {"Method": "PELT", "Input": "Monthly mean prediction loss", "Timing": "Retrospective segment", "Default event count": len(change_dates), "Role": "Describes the completed loss path"},
])
save_table(detector_summary, "detector_method_comparison", "Inputs, timing, and roles of the three change calculations.", "tab:chapter14-detector-comparison", fit=True)

timeline_rows = []
for method, dates in [
    ("Fixed threshold CUSUM", fixed_events),
    ("Changing threshold CUSUM", changing_events),
    ("ADWIN", pd.DatetimeIndex(alarm_dates["Date"])),
    ("PELT", pd.DatetimeIndex(change_dates["First month after change"])),
]:
    timeline_rows.extend({"Date": date, "Method": method} for date in dates)
timeline = pd.DataFrame(timeline_rows)
timeline.to_csv(FIGURE_DATA / "detector_timeline.csv", index=False)
positions = {"Fixed threshold CUSUM": 0, "Changing threshold CUSUM": 1, "ADWIN": 2, "PELT": 3}
colors = {"Fixed threshold CUSUM": PALETTE["amber"], "Changing threshold CUSUM": PALETTE["green"], "ADWIN": PALETTE["rose"], "PELT": PALETTE["violet"]}
figure, axis = plt.subplots(figsize=(11.2, 5.2), constrained_layout=True)
for method, position in positions.items():
    dates = timeline.loc[timeline["Method"] == method, "Date"]
    axis.vlines(dates, position - 0.34, position + 0.34, color=colors[method], linewidth=0.65, alpha=0.75)
axis.set_yticks(list(positions.values()), list(positions.keys()))
axis.set_xlabel("Date")
axis.set_title("Different Inputs Produce Different Change Dates")
axis.set_ylim(-0.7, 3.7)
finish_axis(axis, grid=False)
save_figure(figure, "detector_timeline")
plt.close(figure)

,Method,Input,Timing,Default event count,Role
0,Fixed threshold CUSUM,Daily log price moves,Online event,"1,599",Marks cumulative price moves
1,Changing threshold CUSUM,Daily log price moves,Online event,986,Uses prior 63 day volatility
2,ADWIN,Daily prediction mistakes,Online alarm,9,May trigger a later refit
3,PELT,Monthly mean prediction loss,Retrospective segment,3,Describes the completed loss path
